# W&B Sweeps — Fine-tune the HiLift Transolver++ checkpoint

Sweeps are W&B's hyperparameter search. A **sweep controller** samples
hyperparameters from a config, hands them to an **agent** that runs
one training trial per proposal, and updates its beliefs from the
metric each trial logs back. The Bayesian method used here proposes
new points based on which regions of the space look most promising —
it doesn't sweep exhaustively, so it converges much faster than grid
search on modest trial counts.

This notebook demonstrates the full loop on the HiLift point-cloud
example:

1. **Pull the HiLift champion** from the Registry (produced by
   `experiment_tracking_hilift.ipynb`) — this is the warm start.
2. **Fine-tune** it under different hyperparameter proposals via a
   Bayesian sweep. Architecture is fixed by the loaded `state_dict`;
   only optimiser-side HPs move.
3. **Promote the sweep winner** into a new Collection
   `checkpoints_tuned` under the same `hilift` Registry, so
   downstream consumers can pull it by canonical path.

**Prerequisites** — both must be run first:
- `registry.ipynb` — produces
  `wandb-registry-hilift/pointcloud_demo:production`.
- `experiment_tracking_hilift.ipynb` — produces
  `wandb-registry-hilift/checkpoints:production`.
- Internet — §2 clones `thuml/Transolver_plus` from GitHub if the
  training notebook hasn't already cached it.


## 1. Setup

Third-party imports and the section-local constants — the notebook
stands alone, but every constant that ties us to the HiLift example
lives in the globals cell so it's easy to scan and tweak.


In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import wandb
import os

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# ---- W&B target ----
ENTITY = os.environ.get("WANDB_ENTITY")
PROJECT = os.environ.get("WANDB_PROJECT")

In [ ]:
# ---- Registry paths ----
REGISTRY_NAME = "hilift"
DATASET_COLLECTION = "pointcloud_demo"
BASE_COLLECTION = "checkpoints"
TUNED_COLLECTION = "checkpoints_tuned"

DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
CHECKPOINT_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{BASE_COLLECTION}:production"

# ---- Data + patch config (identical to experiment_tracking_hilift) ----
POINTS_PER_CASE_CACHE = 8192
PATCH_POINTS = 1024
PATCHES_PER_CASE = 96

# ---- Sweep budget ----
# Fine-tune is fast (5 epochs from a warm start); 8 trials keeps the
# whole sweep under ~15 min on a laptop CPU. Bump `SWEEP_COUNT` to
# spend more compute on Bayesian exploration.
FINE_TUNE_EPOCHS = 10
SWEEP_COUNT = 8

TMP_DIR = Path(tempfile.mkdtemp(prefix="hilift_sweep_"))

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"device = {device}")
print(f"tmp    = {TMP_DIR}")


## 2. Transolver++ setup — clone + patch (reuse cache if present)

The training notebook `experiment_tracking_hilift.ipynb` already
clones `thuml/Transolver_plus` into `~/.cache/physicsx_demo/` and
patches its `dist_nn.all_reduce` calls with a `_maybe_all_reduce`
helper that no-ops when no distributed process group is active. This
notebook re-uses that cached, already-patched module — if you've run
the training notebook once, this cell is a no-op.


In [ ]:
CACHE_ROOT = Path.home() / ".cache" / "physicsx_demo"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSOLVER_DIR = CACHE_ROOT / "Transolver_plus"

if not TRANSOLVER_DIR.exists():
    print(f"Cloning thuml/Transolver_plus into {TRANSOLVER_DIR} ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/thuml/Transolver_plus.git",
        str(TRANSOLVER_DIR),
    ])
else:
    print(f"Using cached Transolver_plus at {TRANSOLVER_DIR}")

model_file = TRANSOLVER_DIR / "models" / "Transolver_plus.py"
src_text = model_file.read_text()

if "def _maybe_all_reduce" not in src_text:
    src_text = src_text.replace(
        "import torch.distributed.nn as dist_nn\n",
        "import torch.distributed.nn as dist_nn\n"
        "import torch.distributed as dist\n\n\n"
        "def _maybe_all_reduce(tensor):\n"
        "    if dist.is_available() and dist.is_initialized() "
        "and dist.get_world_size() > 1:\n"
        "        dist_nn.all_reduce(tensor, op=dist_nn.ReduceOp.SUM)\n"
        "    return tensor\n",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_norm, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_norm)",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_token, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_token)",
    )
    model_file.write_text(src_text)
    print("Patched Transolver_plus.py with _maybe_all_reduce.")
else:
    print("Transolver_plus.py already patched.")

if str(TRANSOLVER_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSOLVER_DIR))

if "models.Transolver_plus" in sys.modules:
    importlib.reload(sys.modules["models.Transolver_plus"])

from models.Transolver_plus import Model as TransolverPlusModel

print("Imported TransolverPlusModel OK.")


## 3. Warm-start + dataset + shared helpers

A short `sweep-setup` run does two jobs at once:

1. **Records lineage** — `run.use_artifact(...)` on both the dataset
   and the checkpoint, so the W&B lineage graph shows the sweep's
   setup step sourcing from the Registry entries produced by the
   HiLift setup + training notebooks.
2. **Materialises everything every trial reuses** — the split
   `train_cases` / `val_cases` / `test_cases`, the train-only
   normalisation `stats`, the pretrained `state_dict` +
   `PRETRAINED_ARCH` config dict, and the Dataset / model / evaluate
   helper definitions. These are defined *once* here so the sweep
   agent's `train_fn` can rebuild only the model + optimiser per
   trial and reuse everything else.

We also print a **baseline** — the val loss of the model with warm
weights and no fine-tuning at all. The sweep goal is to beat this.


In [ ]:
# ==== §3 — Pull warm-start + dataset, define shared helpers =========

def _load_npz_case(path: Path) -> dict:
    """Load one setup-notebook NPZ back into a plain-Python dict."""
    data = np.load(path, allow_pickle=True)
    case = {k: data[k] for k in data.files}

    for key in ["geo_id", "run_id", "target_name",
                "target_location", "target_mode"]:
        if key in case:
            case[key] = (
                str(case[key].item()) if case[key].ndim == 0 else str(case[key])
            )
    case["aoa"] = float(np.asarray(case["aoa"]).item())
    return case


def split_cases(
    cases_list: list[dict],
) -> tuple[list[dict], list[dict], list[dict], str]:
    """Case-level 70/15/15 split with a point-level fallback for <3 cases."""
    rng = np.random.default_rng(SEED)

    if len(cases_list) >= 3:
        order = np.arange(len(cases_list))
        rng.shuffle(order)

        n_train = max(1, int(0.70 * len(cases_list)))
        n_val = max(1, int(0.15 * len(cases_list)))

        train = [cases_list[i] for i in order[:n_train]]
        val = [cases_list[i] for i in order[n_train:n_train + n_val]]
        test = [cases_list[i] for i in order[n_train + n_val:]]

        if not test:
            test = val

        return train, val, test, "case split"

    train, val, test = [], [], []
    for case in cases_list:
        n = len(case["points"])
        order = np.arange(n)
        rng.shuffle(order)

        n_train = int(0.70 * n)
        n_val = int(0.15 * n)

        def _slice(idx: np.ndarray, suffix: str) -> dict:
            return {
                **case,
                "points": case["points"][idx],
                "normals": case["normals"][idx],
                "target": case["target"][idx],
                "run_id": f"{case['run_id']}_{suffix}",
            }

        train.append(_slice(order[:n_train], "train_points"))
        val.append(_slice(order[n_train:n_train + n_val], "val_points"))
        test.append(_slice(order[n_train + n_val:], "test_points"))

    return train, val, test, "point split within case(s)"


def aoa_condition(aoa_degrees: float, stats: dict) -> np.ndarray:
    """Encode angle of attack as (z-norm, sin(rad), cos(rad))."""
    aoa_norm = (np.float32(aoa_degrees) - stats["aoa_mean"]) / stats["aoa_std"]
    aoa_rad = np.deg2rad(np.float32(aoa_degrees))
    return np.array(
        [aoa_norm, np.sin(aoa_rad), np.cos(aoa_rad)],
        dtype=np.float32,
    )


class HiLiftPointPatchDataset(Dataset):
    """Yield random `patch_points`-sized subsets of each case per epoch.

    Identical to the training-notebook Dataset. Fresh random patches
    during train, deterministic per-index during val so metrics don't
    jitter epoch-to-epoch.
    """

    def __init__(
        self,
        case_list: list[dict],
        stats: dict,
        patch_points: int,
        patches_per_case: int,
        random_patches: bool = True,
    ) -> None:
        self.case_list = case_list
        self.stats = stats
        self.patch_points = patch_points
        self.patches_per_case = patches_per_case
        self.random_patches = random_patches
        self.rng = np.random.default_rng(SEED)

    def __len__(self) -> int:
        return len(self.case_list) * self.patches_per_case

    def __getitem__(
        self, idx: int,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        case = self.case_list[idx % len(self.case_list)]
        n = len(case["points"])
        replace = n < self.patch_points

        if self.random_patches:
            point_idx = self.rng.choice(n, size=self.patch_points, replace=replace)
        else:
            rng = np.random.default_rng(SEED + idx)
            point_idx = rng.choice(n, size=self.patch_points, replace=replace)

        points = case["points"][point_idx].astype(np.float32)
        normals = case["normals"][point_idx].astype(np.float32)
        target = case["target"][point_idx].astype(np.float32)

        pos = (points - self.stats["point_mean"]) / self.stats["point_std"]
        y = (target - self.stats["target_mean"]) / self.stats["target_std"]

        x = np.concatenate([pos, normals], axis=1).astype(np.float32)
        cond = aoa_condition(case["aoa"], self.stats)

        return (
            torch.tensor(x),
            torch.tensor(pos.astype(np.float32)),
            torch.tensor(cond),
            torch.tensor(y.astype(np.float32)),
        )


class TransolverPointFieldSurrogate(nn.Module):
    """Point-cloud → per-point scalar prediction via Transolver++."""

    def __init__(
        self,
        feature_dim: int = 6,
        out_dim: int = 1,
        n_hidden: int = 96,
        n_layers: int = 4,
        n_head: int = 4,
        slice_num: int = 16,
        mlp_ratio: int = 2,
        dropout: float = 0.05,
    ) -> None:
        super().__init__()
        self.backbone = TransolverPlusModel(
            space_dim=feature_dim,
            fun_dim=0,
            out_dim=out_dim,
            n_hidden=n_hidden,
            n_layers=n_layers,
            n_head=n_head,
            mlp_ratio=mlp_ratio,
            slice_num=slice_num,
            dropout=dropout,
            unified_pos=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        condition: torch.Tensor,
    ) -> torch.Tensor:
        return self.backbone((x, pos, condition))


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    stats: dict,
) -> tuple[float, float, float]:
    """Score `model` over `loader`. Same triple as evaluate_patch_loader in
    the training notebook, but takes `model` + `stats` as explicit args
    so a sweep agent can call it on a fresh trial model."""
    model.eval()
    torch.manual_seed(SEED)

    target_mean_t = torch.tensor(
        stats["target_mean"], dtype=torch.float32, device=device,
    )
    target_std_t = torch.tensor(
        stats["target_std"], dtype=torch.float32, device=device,
    )

    losses: list[float] = []
    maes: list[float] = []
    rmses: list[float] = []

    for x, pos, cond, y in loader:
        x = x.to(device); pos = pos.to(device)
        cond = cond.to(device); y = y.to(device)

        pred = model(x, pos, cond)
        loss = F.mse_loss(pred, y)

        pred_phys = pred * target_std_t + target_mean_t
        y_phys = y * target_std_t + target_mean_t
        err = pred_phys - y_phys

        losses.append(loss.item())
        maes.append(err.abs().mean().item())
        rmses.append(torch.sqrt((err ** 2).mean()).item())

    return float(np.mean(losses)), float(np.mean(maes)), float(np.mean(rmses))


In [ ]:
# ---- One short setup run: record lineage + pull both artifacts ----
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-setup",
    name="hilift-sweep-setup",
    tags=["notebook", "hilift", "sweep", "setup"],
) as setup_run:

    ckpt_art = setup_run.use_artifact(CHECKPOINT_REGISTRY_PATH)
    ckpt_dir = Path(ckpt_art.download())
    ds_art = setup_run.use_artifact(DATASET_REGISTRY_PATH)
    ds_dir = Path(ds_art.download())

    # ---- Load checkpoint dict ----
    ckpt_file = next(ckpt_dir.glob("*.pt"))
    ckpt = torch.load(ckpt_file, map_location=device, weights_only=False)

    PRETRAINED_STATE_DICT = ckpt["state_dict"]
    PRETRAINED_ARCH = {
        "feature_dim": 6,
        "out_dim":     int(ckpt.get("target_dim", 1)),
        "n_hidden":    96,
        "n_layers":    4,
        "n_head":      4,
        "slice_num":   16,
        "mlp_ratio":   2,
    }
    print(
        f"loaded HiLift champion (epoch {ckpt.get('epoch')}, "
        f"val_loss {ckpt.get('val_loss'):.4f})"
    )
    print(f"pretrained arch: {PRETRAINED_ARCH}")

    # ---- Load NPZ cases + split + normalise ----
    cases: list[dict] = [_load_npz_case(p) for p in sorted(ds_dir.glob("*.npz"))]
    train_cases, val_cases, test_cases, split_mode = split_cases(cases)

    train_points_all = np.concatenate([c["points"] for c in train_cases], axis=0)
    train_target_all = np.concatenate([c["target"] for c in train_cases], axis=0)

    stats: dict = {
        "point_mean":  train_points_all.mean(axis=0).astype(np.float32),
        "point_std":   np.maximum(train_points_all.std(axis=0), 1e-6).astype(np.float32),
        "target_mean": train_target_all.mean(axis=0).astype(np.float32),
        "target_std":  np.maximum(train_target_all.std(axis=0), 1e-6).astype(np.float32),
        "aoa_mean":    np.float32(np.mean([c["aoa"] for c in train_cases])),
        "aoa_std":     np.float32(max(np.std([c["aoa"] for c in train_cases]), 1.0)),
    }
    TARGET_DIM = train_cases[0]["target"].shape[1]

    print(
        f"loaded {len(cases)} cases    "
        f"split: {len(train_cases)} train / {len(val_cases)} val / "
        f"{len(test_cases)} test    ({split_mode})"
    )

    # ---- Baseline: val loss of the warm-start model, no fine-tuning ----
    baseline_model = TransolverPointFieldSurrogate(**PRETRAINED_ARCH).to(device)
    baseline_model.load_state_dict(PRETRAINED_STATE_DICT)
    baseline_loader = DataLoader(
        HiLiftPointPatchDataset(
            val_cases, stats, PATCH_POINTS,
            max(16, PATCHES_PER_CASE // 4),
            random_patches=False,
        ),
        batch_size=4, shuffle=False,
    )
    baseline_loss, baseline_mae, baseline_rmse = evaluate(
        baseline_model, baseline_loader, stats,
    )
    del baseline_model
    print(
        f"warm-start baseline: val_metrics/loss = {baseline_loss:.4f}   "
        f"(sweep goal: beat this)"
    )

## 4. Sweep configuration and training function

Four hyperparameters, all optimiser-side — architecture is fixed by
the loaded `state_dict`:

- **`lr`** — dominant fine-tune knob. Log-uniform `[1e-5, 3e-3]`.
- **`weight_decay`** — regularisation on top of already-trained
  weights. Log-uniform `[1e-6, 1e-3]`.
- **`dropout`** — regularisation on the attention / MLP branches.
  `nn.Dropout` has no learned parameters, so it's safe to sweep even
  after `load_state_dict`. Uniform `[0.0, 0.15]`.
- **`batch_size`** — discrete `[4, 8]`. Trades gradient noise for
  step count on the small train split.

The `train_fn` closure captures the pretrained state + normalised
stats + already-split cases from §3, so each trial starts from an
identical warm start without re-downloading or re-splitting.

Architecture HPs (`n_hidden`, `n_layers`, `slice_num`, ...) are
deliberately **not** in the sweep — they'd break `state_dict`
loading. See §7 for how to sweep them from-scratch.


In [ ]:
SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {"name": "val_metrics/loss", "goal": "minimize"},
    "parameters": {
        "lr":           {"distribution": "log_uniform_values", "min": 1e-5, "max": 3e-3},
        "weight_decay": {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-3},
        "dropout":      {"distribution": "uniform",             "min": 0.0,  "max": 0.15},
        "batch_size":   {"values": [4, 8]},
    },
}


def train_fn() -> None:
    """One sweep trial — fine-tune the pretrained checkpoint for a few epochs."""
    with wandb.init(
        job_type="sweep-trial",
        tags=["notebook", "hilift", "sweep", "fine-tune"],
    ) as run:
        cfg = run.config

        # Record source artifacts on this trial too so every trial
        # appears as a consumer of the champion + dataset in the
        # lineage graph.
        run.use_artifact(CHECKPOINT_REGISTRY_PATH)
        run.use_artifact(DATASET_REGISTRY_PATH)

        # Build fresh model with pretrained arch + trial dropout.
        model = TransolverPointFieldSurrogate(
            **{**PRETRAINED_ARCH, "dropout": float(cfg.dropout)},
        ).to(device)
        model.load_state_dict(PRETRAINED_STATE_DICT)

        # Per-trial DataLoaders (batch size sweeps).
        train_loader = DataLoader(
            HiLiftPointPatchDataset(
                train_cases, stats, PATCH_POINTS, PATCHES_PER_CASE,
                random_patches=True,
            ),
            batch_size=int(cfg.batch_size), shuffle=True, drop_last=False,
        )
        val_loader = DataLoader(
            HiLiftPointPatchDataset(
                val_cases, stats, PATCH_POINTS,
                max(16, PATCHES_PER_CASE // 4),
                random_patches=False,
            ),
            batch_size=int(cfg.batch_size), shuffle=False,
        )

        optim = torch.optim.AdamW(
            model.parameters(),
            lr=float(cfg.lr),
            weight_decay=float(cfg.weight_decay),
        )

        val_loss = val_mae = val_rmse = float("nan")
        for epoch in range(1, FINE_TUNE_EPOCHS + 1):
            # -- Train pass --
            model.train()
            train_losses: list[float] = []
            for x, pos, cond, y in train_loader:
                x = x.to(device); pos = pos.to(device)
                cond = cond.to(device); y = y.to(device)

                optim.zero_grad(set_to_none=True)
                pred = model(x, pos, cond)
                loss = F.mse_loss(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                train_losses.append(loss.item())

            # -- Validate + log --
            train_loss_epoch = float(np.mean(train_losses))
            val_loss, val_mae, val_rmse = evaluate(model, val_loader, stats)
            rmse_norm = val_rmse / float(stats["target_std"].mean())

            run.log({
                "epoch": epoch,
                "train/loss": train_loss_epoch,
                "val_metrics/loss": val_loss,
                "val_metrics/mae": val_mae,
                "val_metrics/rmse": val_rmse,
                "val_metrics/rmse_norm": rmse_norm,
                "learning_rate": float(cfg.lr),
            })

        # Save this trial's fine-tuned checkpoint as a model artifact.
        tmp = Path(tempfile.mkdtemp(prefix="hilift_sweep_trial_"))
        ckpt_path = tmp / "hilift_transolverpp_tuned.pt"
        torch.save(
            {
                "state_dict": model.state_dict(),
                "stats": stats,
                "target_dim": TARGET_DIM,
                "arch": PRETRAINED_ARCH,
                "epochs_fine_tuned": FINE_TUNE_EPOCHS,
                "final_val_loss": val_loss,
                "sweep_hyperparameters": dict(cfg),
            },
            ckpt_path,
        )
        art = wandb.Artifact(
            name="hilift_transolverpp_ckpt_tuned",
            type="model",
            metadata={
                "val_loss": val_loss,
                "val_mae":  val_mae,
                "val_rmse": val_rmse,
                **{k: float(v) if isinstance(v, (int, float)) else v
                   for k, v in dict(cfg).items()},
            },
        )
        art.add_file(str(ckpt_path))
        run.log_artifact(art)

        run.summary.update({
            "final_val_loss": val_loss,
            "final_val_mae":  val_mae,
            "final_val_rmse": val_rmse,
        })


## 5. Launch the sweep

`wandb.sweep` registers the config with the controller and returns a
`sweep_id`. `wandb.agent` polls the controller for HP proposals and
runs `train_fn` once per proposal, up to `SWEEP_COUNT` times. The
controller uses the `val_metrics/loss` each trial logs to update its
Bayesian model of the search space.

Open the printed URL in a browser to watch the sweep populate live —
parallel-coordinates chart, parameter-importance panel, per-trial
curves.


In [ ]:
sweep_id = wandb.sweep(SWEEP_CONFIG, entity=ENTITY, project=PROJECT)

print(f"sweep id  = {sweep_id}")
print(f"sweep url = https://wandb.ai/{ENTITY}/{PROJECT}/sweeps/{sweep_id}")

wandb.agent(sweep_id, function=train_fn, count=SWEEP_COUNT)


## 6. Promote the winner into a new Registry Collection

The best trial is the one with the lowest final `val_metrics/loss`.
We resolve it via `wandb.Api().sweep(...)`, grab its logged model
artifact, and `run.link_artifact` it into a new Collection
`checkpoints_tuned` under the existing `hilift` Registry — no
`create_registry` call needed; Collections are created implicitly
on first link.

Two aliases go on the winner:
- `production` — the canonical alias downstream consumers resolve.
- `sweep-winner` — a stable pointer at *this specific* sweep's
  champion. Useful if you re-sweep with different HP ranges later
  and want to compare winners across sweeps.


In [ ]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-promotion",
    name="hilift-sweep-winner-promotion",
    tags=["hilift", "sweep", "promotion"],
) as promo_run:

    api = wandb.Api()
    sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sweep_id}")

    # Best trial by final val_metrics/loss (min).
    best = min(
        sweep.runs,
        key=lambda r: r.summary.get("val_metrics/loss", float("inf")),
    )
    best_loss = best.summary.get("val_metrics/loss")
    print(
        f"winner: {best.name}   val_metrics/loss = {best_loss:.4f}   "
        f"(baseline was {baseline_loss:.4f})"
    )

    # Grab the model artifact this trial logged.
    best_art = next(a for a in best.logged_artifacts() if a.type == "model")

    target_path = f"wandb-registry-{REGISTRY_NAME}/{TUNED_COLLECTION}"
    promo_run.link_artifact(
        best_art,
        target_path=target_path,
        aliases=["production", "sweep-winner"],
    )

    print(f"linked {best_art.name}:{best_art.version} -> {target_path}")
    print(f"aliases: production, sweep-winner")


## 7. Wrap up — what to look at, and how does this generalise

**Sweep page** (URL printed in §5)
- Parallel-coordinates plot with all four HPs and the final metric.
- Parameter-importance panel — expect `lr` to dominate.
- 8 individual trial runs, each with its `train/loss` and
  `val_metrics/*` curves.

**Lineage graph** (Artifacts tab)
- Every sweep trial run consumes both Registry entries:
  `hilift/checkpoints:production` and
  `hilift/pointcloud_demo:production`. Setup + promotion runs
  consume the same two. This means the tuned model traces cleanly
  back through its parent training run to the underlying dataset.

**Registry**
- Registry `hilift` now has three Collections:
  - `pointcloud_demo` — dataset (from the setup notebook).
  - `checkpoints` — base model versions (from the training
    notebook).
  - `checkpoints_tuned` — the sweep winner, aliased `production` +
    `sweep-winner`.

Downstream notebooks / consumers can pull the tuned model with
`run.use_artifact("wandb-registry-hilift/checkpoints_tuned:production")`.

---

### How does this generalise?

This notebook shows the smallest useful sweep — one architecture,
one dataset, one warm start, four HPs. The same three-act pattern
(consume from Registry → sweep → promote to a new Collection)
extends cleanly along several axes:

**Multiple architectures.** Drop the `state_dict` load. Add the
architecture HPs to the sweep config:

```python
"parameters": {
    "n_hidden":  {"values": [64, 96, 128]},
    "n_layers":  {"values": [3, 4, 6]},
    "slice_num": {"values": [8, 16, 32]},
    ...  # lr, weight_decay, dropout, batch_size still swept
}
```

Each trial now builds a fresh Transolver++ with its own arch and
trains from scratch (10–15 epochs). Sweep goal metric unchanged —
`val_metrics/loss`. The controller now searches jointly over
architecture and optimiser HPs, so parameter-importance in the UI
tells you which arch dimensions matter for this dataset. Promote the
winner into a new Collection `checkpoints_arch_search` for clarity.

**Multiple teams.** Each team publishes its base checkpoint under
its own alias on the same `checkpoints` Collection —
`checkpoints:aero_v3`, `checkpoints:aero_v4`,
`checkpoints:combustion_v1`. The sweep notebook parameterises the
warm-start path:

```python
BASE_ALIAS = "aero_v3"                # per-team knob
CHECKPOINT_REGISTRY_PATH = (
    f"wandb-registry-hilift/checkpoints:{BASE_ALIAS}"
)
```

and promotes the winner under a matching team-scoped alias on
`checkpoints_tuned`. Result: each team gets an independently
Bayesian-tuned model without needing its own Registry.

**Multiple experiments.** Nest sweeps. The outer loop varies the
base alias; the inner loop runs a small optimiser-HP sweep for each
base. After all outer × inner trials complete,
`wandb.Api().sweeps(...)` gives you a comparable-across-bases
summary — which base checkpoint responds best to fine-tuning, which
HP ranges transfer, which don't. The Registry lineage graph makes
the whole nesting legible: click any tuned model, walk back to its
base, walk back further to the dataset.

All three extensions reuse the same `train_fn` skeleton — only the
sweep config, warm-start resolution, and promote-target vary.
